# Master Pipeline Health Checks
Orchestrates data quality checks across all medallion architecture layers (Bronze → Silver → Gold)

In [0]:
from datetime import datetime
import json

print("="*70)
print("🏥 MASTER PIPELINE HEALTH CHECK ORCHESTRATOR")
print("="*70)
print(f"\n📅 Run Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Configuration
layers = ['bronze', 'silver', 'gold']
health_checker_path = "/Users/jiasheng.chen@u.nus.edu/pipeline_health_checker"
timeout_seconds = 300

# Results tracker
all_results = {}

In [0]:
# Run health checks for each layer
for layer in layers:
    print(f"\n{'='*70}")
    print(f"🔍 Running {layer.upper()} Layer Health Checks")
    print(f"{'='*70}\n")
    
    try:
        # Run the health checker notebook with layer parameter
        result = dbutils.notebook.run(
            health_checker_path,
            timeout_seconds=timeout_seconds,
            arguments={"layer": layer}
        )
        
        all_results[layer] = {
            "status": "success",
            "result": result,
            "timestamp": datetime.now().isoformat()
        }
        
        print(f"✅ {layer.upper()} layer checks completed successfully")
        
    except Exception as e:
        all_results[layer] = {
            "status": "failed",
            "error": str(e),
            "timestamp": datetime.now().isoformat()
        }
        print(f"❌ {layer.upper()} layer checks failed: {str(e)}")

print(f"\n{'='*70}")
print("🏁 All health checks completed")
print(f"{'='*70}\n")

In [0]:
# Generate summary report
print("\n" + "="*70)
print("📊 HEALTH CHECK SUMMARY")
print("="*70 + "\n")

total_layers = len(layers)
successful_layers = sum(1 for r in all_results.values() if r["status"] == "success")
failed_layers = total_layers - successful_layers

print(f"Total Layers Checked: {total_layers}")
print(f"✅ Successful: {successful_layers}")
print(f"❌ Failed: {failed_layers}")
print(f"\nSuccess Rate: {(successful_layers/total_layers)*100:.1f}%\n")

# Detailed results
for layer, result in all_results.items():
    status_icon = "✅" if result["status"] == "success" else "❌"
    print(f"{status_icon} {layer.upper()}: {result['status']}")
    if result["status"] == "failed":
        print(f"   Error: {result['error']}")

# Save results to JSON for tracking
results_json = json.dumps(all_results, indent=2)
print(f"\n\n📋 Detailed Results:\n{results_json}")

In [0]:
%sql
-- View historical health check results
SELECT 
  timestamp,
  layer,
  table_name,
  check_type,
  passed,
  severity,
  message
FROM workspace.default.data_quality_results
WHERE timestamp >= current_date() - INTERVAL 7 DAYS
ORDER BY timestamp DESC, layer, table_name
LIMIT 100